# Week 02 — Data Cleaning & Feature Engineering

Solutions for `23CSE301_ML_Week2_Guide.pdf`. Builds on Week 1.

As instructed: **Iris** powers the examples, **Wine** powers every exercise. The messy/clean dataset generators from the guide are defined once below.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, load_wine

pd.set_option('display.width', 120)

## Setup — messy dataset generators (Part I)

Fixed seeds make the 'mess' reproducible.

In [ ]:
def make_dirty_iris(seed=0):
    iris = load_iris()
    df = pd.DataFrame(iris.data,
        columns=['sepal_length','sepal_width','petal_length','petal_width'])
    df['target'] = iris.target
    rng = np.random.default_rng(seed)
    sites = np.array(['North', 'South', 'East'])
    region = sites[rng.integers(0, 3, size=len(df))]
    variants = {'North': ['North', ' north', 'NORTH', 'Nort'],
                'South': ['South', 'south ', 'SOUTH', 'South'],
                'East':  ['East', ' east', 'EAST', 'Esat']}
    df['region'] = [rng.choice(variants[r]) for r in region]
    base = pd.Timestamp('2023-01-01')
    offsets = rng.integers(0, 300, size=len(df))
    df['collection_date'] = [str((base + pd.Timedelta(days=int(o))).date()) for o in offsets]
    for col in ['sepal_length', 'sepal_width']:
        miss_idx = rng.choice(df.index, size=6, replace=False)
        df.loc[miss_idx, col] = np.nan
    out_idx = rng.choice(df.index, size=3, replace=False)
    df.loc[out_idx, 'sepal_length'] = df.loc[out_idx, 'sepal_length'] * 6
    pl = df['petal_length'].astype(str)
    junk_idx = rng.choice(df.index, size=4, replace=False)
    pl.loc[junk_idx] = 'unknown'
    space_idx = rng.choice(df.index, size=4, replace=False)
    pl.loc[space_idx] = pl.loc[space_idx].apply(lambda v: f' {v} ')
    df['petal_length'] = pl
    dup_idx = rng.choice(df.index, size=5, replace=False)
    df = pd.concat([df, df.loc[dup_idx]], ignore_index=True)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def make_dirty_wine(seed=1):
    wine = load_wine()
    df = pd.DataFrame(wine.data, columns=wine.feature_names)
    df['target'] = wine.target
    rng = np.random.default_rng(seed)
    regions = np.array(['Tuscany', 'Piedmont', 'Veneto'])
    region = regions[rng.integers(0, 3, size=len(df))]
    variants = {'Tuscany': ['Tuscany', ' tuscany', 'TUSCANY', 'Toscany'],
                'Piedmont': ['Piedmont', 'piedmont ', 'PIEDMONT', 'Piedmont'],
                'Veneto': ['Veneto', ' veneto', 'VENETO', 'Veneto']}
    df['storage_region'] = [rng.choice(variants[r]) for r in region]
    base = pd.Timestamp('2022-09-01')
    offsets = rng.integers(0, 120, size=len(df))
    df['bottling_date'] = [str((base + pd.Timedelta(days=int(o))).date()) for o in offsets]
    for col in ['magnesium', 'proline']:
        miss_idx = rng.choice(df.index, size=7, replace=False)
        df.loc[miss_idx, col] = np.nan
    out_idx = rng.choice(df.index, size=4, replace=False)
    df.loc[out_idx, 'color_intensity'] = df.loc[out_idx, 'color_intensity'] * 5
    av = df['ash'].astype(str)
    junk_idx = rng.choice(df.index, size=5, replace=False)
    av.loc[junk_idx] = 'NA'
    space_idx = rng.choice(df.index, size=5, replace=False)
    av.loc[space_idx] = av.loc[space_idx].apply(lambda v: f' {v} ')
    df['ash'] = av
    dup_idx = rng.choice(df.index, size=6, replace=False)
    df = pd.concat([df, df.loc[dup_idx]], ignore_index=True)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

# Part I — Data Cleaning

## 1.1 Diagnosing Data Quality Issues

In [ ]:
dfw = make_dirty_wine()
dfw.info()
print('\nMissing per column:')
print(dfw.isnull().sum()[dfw.isnull().sum() > 0])
print('\nDuplicate rows:', dfw.duplicated().sum())
# 1. magnesium & proline have 7 missing each; 6 duplicate rows; 'ash' is object
#    (text) and 'bottling_date'/'storage_region' are text too.
# 2. color_intensity has an unbelievable maximum (outliers injected at x5):
ci = dfw['color_intensity']
print('\ncolor_intensity  max=%.2f  median=%.2f  -> max is %.1fx the median'
      % (ci.max(), ci.median(), ci.max() / ci.median()))
# 3. distinct spellings of what should be 3 regions:
print('\nstorage_region spellings:', dfw['storage_region'].unique())
print('distinct spellings:', dfw['storage_region'].nunique())

## 1.2 Removing Duplicate Records

In [ ]:
dfw = make_dirty_wine()
# 1. count + remove exact duplicates
print('duplicates:', dfw.duplicated().sum())
print('before:', dfw.shape, '-> after:', dfw.drop_duplicates().shape)
# 2. keep='last' removes the same NUMBER of rows, so the shape is identical;
#    only WHICH copy is kept differs.
print('keep=last:', dfw.drop_duplicates(keep='last').shape)
# 3. subset on two columns can drop MORE rows: two wines may share alcohol &
#    color_intensity by chance without being full-row duplicates.
print('subset:', dfw.drop_duplicates(subset=['alcohol', 'color_intensity']).shape)

## 1.3 Fixing Data Types

In [ ]:
dfw = make_dirty_wine()
# 1. clean ash: strip whitespace then coerce to numeric
ash = pd.to_numeric(dfw['ash'].str.strip(), errors='coerce')
dfw['ash'] = ash
print('ash dtype:', dfw['ash'].dtype, '| became NaN:', dfw['ash'].isnull().sum())
# 2. bottling_date -> datetime
dfw['bottling_date'] = pd.to_datetime(dfw['bottling_date'])
print('bottling_date dtype:', dfw['bottling_date'].dtype)
# 3. target -> category
dfw['target'] = dfw['target'].astype('category')
print('target categories:', list(dfw['target'].cat.categories))

## 1.4 Imputing Missing Values

In [ ]:
dfw = make_dirty_wine()
# 1. magnesium -> median, proline -> mean
dfw['magnesium'] = dfw['magnesium'].fillna(dfw['magnesium'].median())
dfw['proline'] = dfw['proline'].fillna(dfw['proline'].mean())
print('missing after impute:', dfw[['magnesium', 'proline']].isnull().sum().to_dict())

# 2. group-wise median (per target class) vs overall median
dfw2 = make_dirty_wine()
overall = dfw2['magnesium'].median()
grouped = dfw2.groupby('target')['magnesium'].transform(lambda s: s.fillna(s.median()))
print('overall median: %.2f | group-wise medians: %s'
      % (overall, dfw2.groupby('target')['magnesium'].median().round(2).to_dict()))

# 3. SimpleImputer(most_frequent) on a categorical column
from sklearn.impute import SimpleImputer
dfw3 = make_dirty_wine()
dfw3.loc[0, 'storage_region'] = np.nan
imp = SimpleImputer(strategy='most_frequent')
dfw3['storage_region'] = imp.fit_transform(dfw3[['storage_region']]).ravel()
print('storage_region missing after impute:', dfw3['storage_region'].isnull().sum())

## 1.5 Detecting Outliers

In [ ]:
dfw = make_dirty_wine()
dfw['magnesium'] = dfw['magnesium'].fillna(dfw['magnesium'].median())
dfw['proline'] = dfw['proline'].fillna(dfw['proline'].median())
col = dfw['color_intensity']
# 1. IQR method
Q1, Q3 = col.quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
iqr_out = dfw[(col < lower) | (col > upper)]
print('IQR bounds: (%.2f, %.2f) | flagged: %d' % (lower, upper, len(iqr_out)))
# 2. Z-score, threshold 3
z = (col - col.mean()) / col.std()
print('Z>3 flagged:', int((z.abs() > 3).sum()))
# 3. stricter Z-score 2.5 -> flags MORE (a lower threshold is easier to exceed)
print('Z>2.5 flagged:', int((z.abs() > 2.5).sum()))

## 1.6 Treating Outliers

In [ ]:
dfw = make_dirty_wine()
dfw['magnesium'] = dfw['magnesium'].fillna(dfw['magnesium'].median())
dfw['proline'] = dfw['proline'].fillna(dfw['proline'].median())
col = dfw['color_intensity']
Q1, Q3 = col.quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
# 1. cap (winsorize)
dfw['ci_capped'] = col.clip(lower, upper)
print('max before: %.2f | after cap: %.2f (upper bound %.2f)'
      % (col.max(), dfw['ci_capped'].max(), upper))
# 2. remove outlier rows
kept = dfw[(col >= lower) & (col <= upper)]
print('rows: %d -> %d after removal' % (len(dfw), len(kept)))
# 3. For this small dataset, CAP is preferred: every sample is valuable, and
#    capping keeps the row while pulling the extreme value into a believable range.

## 1.7 Cleaning Messy Text & Categorical Values

In [ ]:
dfw = make_dirty_wine()
# 1. every distinct spelling
print('raw spellings:', dfw['storage_region'].unique())
# 2. normalize case/whitespace, then map typos to 3 canonical categories
norm = dfw['storage_region'].str.strip().str.lower()
typo_map = {
    'tuscany': 'tuscany', 'toscany': 'tuscany',
    'piedmont': 'piedmont',
    'veneto': 'veneto',
}
dfw['storage_region_clean'] = norm.map(typo_map)
# 3. confirm zero leftover NaN
print('cleaned categories:', dfw['storage_region_clean'].unique())
print('unmapped (NaN):', dfw['storage_region_clean'].isnull().sum())

# Part II — Feature Engineering

Clean FE-ready dataset generators (from the guide):

In [ ]:
def make_clean_iris_fe(seed=0):
    iris = load_iris()
    df = pd.DataFrame(iris.data,
        columns=['sepal_length','sepal_width','petal_length','petal_width'])
    df['target'] = iris.target
    rng = np.random.default_rng(seed)
    sites = np.array(['North', 'South', 'East'])
    df['region'] = sites[rng.integers(0, 3, size=len(df))]
    base = pd.Timestamp('2023-01-01')
    offsets = rng.integers(0, 300, size=len(df))
    df['collection_date'] = pd.to_datetime([base + pd.Timedelta(days=int(o)) for o in offsets])
    return df


def make_clean_wine_fe(seed=1):
    wine = load_wine()
    df = pd.DataFrame(wine.data, columns=wine.feature_names)
    df['target'] = wine.target
    rng = np.random.default_rng(seed)
    regions = np.array(['Tuscany', 'Piedmont', 'Veneto'])
    df['storage_region'] = regions[rng.integers(0, 3, size=len(df))]
    base = pd.Timestamp('2022-09-01')
    offsets = rng.integers(0, 120, size=len(df))
    df['bottling_date'] = pd.to_datetime([base + pd.Timedelta(days=int(o)) for o in offsets])
    return df

WINE_NUM = list(load_wine().feature_names)  # the 13 numeric features

## 2.1 Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
df = make_clean_wine_fe()
# 1. StandardScaler -> mean ~0
std = StandardScaler().fit_transform(df[WINE_NUM])
print('StandardScaler column means ~', np.round(std.mean(axis=0), 2)[:5], '...')
# 2. MinMaxScaler -> [0, 1]
mm = MinMaxScaler().fit_transform(df[WINE_NUM])
print('MinMax ranges: min=%.1f max=%.1f' % (mm.min(), mm.max()))
# 3. RobustScaler vs StandardScaler on proline (has extreme values)
rob = RobustScaler().fit_transform(df[['proline']])
sc = StandardScaler().fit_transform(df[['proline']])
print('proline range  robust: %.2f  standard: %.2f' % (np.ptp(rob), np.ptp(sc)))
# RobustScaler uses median/IQR, so extreme values inflate its range less.

## 2.2 Encoding Categorical Variables (scikit-learn)

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
df = make_clean_wine_fe()
# 1. one-hot encode storage_region
ohe = OneHotEncoder(sparse_output=False)
region_ohe = ohe.fit_transform(df[['storage_region']])
print('OHE shape:', region_ohe.shape, '| names:', list(ohe.get_feature_names_out()))
# 2. ordinal-encode a ranked quality column low < medium < high
quality = pd.DataFrame({'quality': ['low', 'high', 'medium', 'low', 'high']})
oe = OrdinalEncoder(categories=[['low', 'medium', 'high']])
print('ordinal codes:', oe.fit_transform(quality).ravel())
# 3. Using OrdinalEncoder on storage_region would invent a false order
#    (Tuscany<Piedmont<Veneto), implying a ranking that does not exist.

## 2.3 Binning Continuous Variables

In [ ]:
df = make_clean_wine_fe()
# 1. equal-width bins on alcohol
df['alcohol_bin'] = pd.cut(df['alcohol'], bins=3, labels=['low', 'medium', 'high'])
print(df['alcohol_bin'].value_counts().sort_index().to_dict())
# 2. equal-frequency bins on color_intensity
df['ci_q'] = pd.qcut(df['color_intensity'], q=4, labels=False)
print('qcut group sizes:', df['ci_q'].value_counts().sort_index().to_dict())
# 3. qcut is a worse choice than cut when you need fixed, interpretable
#    thresholds (e.g. medical cutoffs) rather than data-driven quantiles.

## 2.4 Extracting Features from Dates

In [ ]:
df = make_clean_wine_fe()
# 1. month + day-of-week
df['bottling_month'] = df['bottling_date'].dt.month
df['bottling_dow'] = df['bottling_date'].dt.dayofweek
# 2. is_weekend + weekend fraction
df['is_weekend'] = df['bottling_dow'].isin([5, 6]).astype(int)
print('weekend fraction: %.3f' % df['is_weekend'].mean())
# 3. days since the campaign start
df['days_since_start'] = (df['bottling_date'] - pd.Timestamp('2022-09-01')).dt.days
print(df[['bottling_month', 'bottling_dow', 'is_weekend', 'days_since_start']].head(3))

## 2.5 Transforming Skewed Features

In [ ]:
from sklearn.preprocessing import PowerTransformer
df = make_clean_wine_fe()
# 1. most skewed numeric column
skews = df[WINE_NUM].skew().abs().sort_values(ascending=False)
worst = skews.index[0]
print('most skewed:', worst, '(skew %.2f)' % df[worst].skew())
# 2. log1p
print('after log1p: %.2f' % np.log1p(df[worst]).skew())
# 3. PowerTransformer (Yeo-Johnson)
pt = PowerTransformer(method='yeo-johnson')
pt_col = pt.fit_transform(df[[worst]]).ravel()
print('after Yeo-Johnson: %.2f' % pd.Series(pt_col).skew())

## 2.6 Interaction & Polynomial Features

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
df = make_clean_wine_fe()
# 1. manual interaction
df['phenol_density'] = df['flavanoids'] / df['total_phenols']
print('phenol_density head:', df['phenol_density'].head(3).round(3).tolist())
# 2. degree-2 polynomial features on two columns
poly2 = PolynomialFeatures(degree=2, include_bias=False)
p2 = poly2.fit_transform(df[['alcohol', 'malic_acid']])
print('degree2 -> %d cols: %s' % (p2.shape[1], list(poly2.get_feature_names_out(['alcohol', 'malic_acid']))))
# 3. degree-3
poly3 = PolynomialFeatures(degree=3, include_bias=False)
p3 = poly3.fit_transform(df[['alcohol', 'malic_acid']])
print('degree3 -> %d cols' % p3.shape[1])
# High degree x many columns explodes the feature count and invites overfitting.

## 2.7 Quick Feature Selection

In [ ]:
from sklearn.feature_selection import VarianceThreshold
df = make_clean_wine_fe()
# 1. variance threshold
vt = VarianceThreshold(threshold=0.5).fit(df[WINE_NUM])
survivors = list(np.array(WINE_NUM)[vt.get_support()])
print('survive var>0.5:', survivors)
# 2. top-3 by absolute correlation with target
corr = df[WINE_NUM + ['target']].corr()['target'].drop('target').abs().sort_values(ascending=False)
print('top 3 corr with target:', list(corr.head(3).index))
# 3. low-variance AND weakly correlated -> first candidates to drop
low_var = set(np.array(WINE_NUM)[~vt.get_support()])
weak = set(corr[corr < 0.3].index)
print('drop-first candidates:', sorted(low_var & weak))

# Part III — Capstone: End-to-End Cleaning & Feature Engineering (Wine)

One pipeline from `make_dirty_wine()` to a fully clean, model-ready DataFrame.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = make_dirty_wine()

# 1. Diagnose
print('missing:', int(df.isnull().sum().sum()), '| duplicates:', int(df.duplicated().sum()))

# 2. Deduplicate
df = df.drop_duplicates().reset_index(drop=True)

# 3. Fix types
df['ash'] = pd.to_numeric(df['ash'].str.strip(), errors='coerce')
df['bottling_date'] = pd.to_datetime(df['bottling_date'])

# 4. Clean text -> 3 canonical regions
norm = df['storage_region'].str.strip().str.lower()
df['storage_region'] = norm.map({'tuscany': 'tuscany', 'toscany': 'tuscany',
                                 'piedmont': 'piedmont', 'veneto': 'veneto'})

# 5. Impute all numeric NaN (magnesium, proline, and ash from step 3) with medians
num_cols = list(load_wine().feature_names)
for c in num_cols:
    if df[c].isnull().any():
        df[c] = df[c].fillna(df[c].median())

# 6. Treat outliers: cap color_intensity at IQR bounds
Q1, Q3 = df['color_intensity'].quantile([0.25, 0.75])
IQR = Q3 - Q1
df['color_intensity'] = df['color_intensity'].clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

# 7. Engineer features
df['phenol_density'] = df['flavanoids'] / df['total_phenols']
df['bottling_month'] = df['bottling_date'].dt.month
df['bottling_dow'] = df['bottling_date'].dt.dayofweek
df['alcohol_bin'] = pd.cut(df['alcohol'], bins=3, labels=['low', 'medium', 'high'])
num_cols = num_cols + ['phenol_density']

# 8. Encode & scale
ohe = OneHotEncoder(sparse_output=False)
region_ohe = pd.DataFrame(ohe.fit_transform(df[['storage_region']]),
                          columns=ohe.get_feature_names_out(), index=df.index)
scaled = pd.DataFrame(StandardScaler().fit_transform(df[num_cols]),
                      columns=num_cols, index=df.index)

# 9. Select: top-3 corr with target; low-variance candidate
corr = df[num_cols + ['target']].corr()['target'].drop('target').abs().sort_values(ascending=False)
print('top 3 corr with target:', list(corr.head(3).index))
variances = df[num_cols].var().sort_values()
print('lowest-variance feature:', variances.index[0])

# 10. Assemble final processed frame + confirm no missing values
processed = pd.concat([
    scaled,
    region_ohe,
    df[['bottling_month', 'bottling_dow', 'alcohol_bin', 'target']],
], axis=1)
print('final shape:', processed.shape)
assert processed.isnull().sum().sum() == 0
print('no missing values remain ->', processed.isnull().sum().sum() == 0)